# Commune Quickstart — Give Your Agent an Inbox in 5 Minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/quickstart.ipynb)

This notebook shows you how to give a Python AI agent its own email address, send your first email, and read a thread — in under 5 minutes.

**What you'll learn:**
- Create a dedicated inbox for your agent
- Send an email from the inbox
- Read conversation threads
- Set up webhook verification

**Prerequisites:** A [Commune API key](https://commune.email) (free tier available, no credit card required)

In [ ]:
# Install the Commune Python SDK
# This takes ~10 seconds
!pip install commune-mail -q
print("\u2713 commune-mail installed")

## Step 1: Initialize the client

Set your API key. Get one at [commune.email](https://commune.email) — it starts with `comm_`.

In production, use environment variables. For this notebook, paste your key directly.

In [ ]:
import os
from commune import CommuneClient

# Replace with your actual API key, or set COMMUNE_API_KEY environment variable
API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")

client = CommuneClient(api_key=API_KEY)
print("\u2713 Client initialized")

## Step 2: Create an inbox for your agent

An inbox is a real, deliverable email address. No domain setup or DNS configuration needed — Commune provides a `@agents.commune.email` subdomain automatically.

Each agent in your system should have its own inbox. This keeps conversation threads isolated per agent and makes it easy to route inbound emails to the right handler.

In [ ]:
# Create an inbox — Commune auto-assigns the domain
# local_part is the part before the @ symbol
inbox = client.inboxes.create(local_part="my-agent")

print(f"\u2713 Inbox created!")
print(f"  Address: {inbox.address}")  # → my-agent@agents.commune.email
print(f"  Inbox ID: {inbox.id}")      # → i_abc123 (use this in API calls)

## Step 3: Send your first email

Use `messages.send()` to send from the inbox. The `inbox_id` tells Commune which inbox to send from (so the "From" address is your agent's address).

For replies, always pass `thread_id` — this keeps the conversation threaded in the recipient's email client (Gmail, Outlook, etc.).

In [ ]:
# Send an email from the inbox
# Replace 'recipient@example.com' with an email you can check
result = client.messages.send(
    to="recipient@example.com",  # ← change this
    subject="Hello from my agent",
    text="Hi! This email was sent by an AI agent using commune-mail.",
    html="<p>Hi! This email was sent by an AI agent using <strong>commune-mail</strong>.</p>",
    inbox_id=inbox.id,
)

print(f"\u2713 Email sent!")
print(f"  Message ID: {result.get('message_id', 'sent')}")

## Step 4: Read conversation threads

Threads group related emails together. When a user replies to your agent's email, the reply appears in the same thread.

`threads.list()` returns the most recent threads. `threads.messages()` returns all messages in a specific thread.

In [ ]:
# List the most recent threads in the inbox
threads = client.threads.list(inbox_id=inbox.id, limit=5)

print(f"\u2713 Found {len(threads.data)} thread(s)")
print()

for thread in threads.data:
    print(f"Thread: {thread.subject or '(no subject)'}")
    print(f"  Messages: {thread.message_count}")
    print(f"  Last activity: {thread.last_message_at}")
    print(f"  Thread ID: {thread.thread_id}")
    print()

In [ ]:
# If there are threads, read the messages in the first one
if threads.data:
    thread_id = threads.data[0].thread_id
    messages = client.threads.messages(thread_id)

    print(f"Messages in thread '{threads.data[0].subject}':")
    for msg in messages:
        sender = next(
            (p.identity for p in msg.participants if p.role == "sender"),
            "unknown"
        )
        direction = "\u2192 Outbound" if msg.direction == "outbound" else "\u2190 Inbound"
        print(f"\n{direction} | From: {sender}")
        print(f"  {msg.content[:200]}...")
else:
    print("No threads yet. Send a reply to the email above and re-run this cell.")

## Step 5: Webhook verification (for production)

When an email arrives at your inbox, Commune sends a POST request to your webhook URL. Always verify the signature to ensure the request came from Commune.

This is important for security — an unverified webhook endpoint could receive spoofed requests.

In [ ]:
from commune import verify_signature, WebhookVerificationError

# Example: verifying a webhook in Flask or FastAPI
example_code = '''
from commune import verify_signature, WebhookVerificationError

@app.post("/webhook")
async def handle_email(request: Request):
    body = await request.body()

    try:
        verify_signature(
            payload=body,
            signature=request.headers["x-commune-signature"],
            secret=os.environ["COMMUNE_WEBHOOK_SECRET"],
            timestamp=request.headers["x-commune-timestamp"],
        )
    except WebhookVerificationError as e:
        return Response(status_code=401, content=str(e))

    payload = json.loads(body)
    thread_id = payload["thread_id"]
    content = payload["content"]

    # Run your agent here, then reply in thread
    agent_reply = your_llm.complete(f"Reply to: {content}")

    client.messages.send(
        to=payload["sender"],
        text=agent_reply,
        inbox_id=payload["inboxId"],
        thread_id=thread_id,
    )

    return {"ok": True}
'''

print("Example webhook handler:")
print(example_code)

## Common Mistake 1: Omitting `thread_id` When Replying

The most impactful single line in any email-agent codebase is `thread_id=...`. Leaving it out seems harmless — `messages.send()` still succeeds — but the consequences are invisible until your users complain.

**Exact failure:** Email clients use the `In-Reply-To` header to group messages into a conversation. Commune only sets `In-Reply-To` when you pass `thread_id`. Without it, Gmail, Outlook, and Apple Mail each render your agent's reply as an unrelated new email — separate subject line, separate notification, no quoted history. A five-email support exchange looks like five cold-outreach emails.

**How to spot it in production:** Look for threads with `message_count == 1`. If your inbox has dozens of single-message threads that should be multi-turn conversations, `thread_id` is almost certainly missing somewhere.


In [ ]:
# ❌ WRONG: Sending a reply without thread_id
# This opens a brand-new thread every time, even for an ongoing conversation.
# The customer's email client treats each message as an unrelated email.

result = client.messages.send(
    to="customer@example.com",
    subject="Re: Your refund request",
    text="Hi! We've processed your refund. It will appear in 3-5 business days.",
    inbox_id=inbox.id,
    # thread_id is missing — Commune creates a fresh thread for this message
)

# Side-effects:
# 1. Customer gets a new email with no quoted history
# 2. Your inbox fills with single-message threads instead of conversations
# 3. If the customer replies to the correct thread, the next webhook payload
#    carries the real thread_id — but the reply you sent lives in a different one


In [ ]:
# ✅ CORRECT: Pass thread_id to keep the conversation in one place
# thread_id comes from:
#   - The inbound webhook payload:  payload["thread_id"]
#   - A threads.list() result:      thread.thread_id
#   - A previous messages.send():   result["thread_id"]

# From a webhook payload (most common case):
thread_id = "thread_abc123"   # replace with payload["thread_id"] in production

result = client.messages.send(
    to="customer@example.com",
    subject="Re: Your refund request",   # subject is optional for replies; Commune inherits it
    text="Hi! We've processed your refund. It will appear in 3-5 business days.",
    inbox_id=inbox.id,
    thread_id=thread_id,   # ← this is the only change — but it's the critical one
)

print(f"Reply sent: {result.get('message_id')}")
# The customer now sees ONE conversation in Gmail, with your reply nested under
# their original message, with full quoted history.


## Common Mistake 2: Polling for New Emails Instead of Using Webhooks

Polling (`threads.list()` in a loop) is the first approach most developers reach for — it's familiar, requires no server setup, and works in a notebook. For a production agent it's the wrong pattern.

**Why polling fails:**
- **Latency:** A 10-second poll interval means customers wait up to 10 s for   a response. A 1-second interval wastes 99 % of API calls on empty results.
- **Rate limits:** Continuous polling burns through your API quota even when   there's nothing to do.
- **Missed messages:** If two emails arrive between polls, you process both in   the same iteration with no ordering guarantees.
- **No push context:** The webhook payload includes sender, subject, content,   and thread_id in one call. Polling requires a second API call to fetch message   content.

**The right pattern:** Register a webhook URL in the Commune dashboard. Commune calls your endpoint within milliseconds of delivery — zero polling, zero wasted calls, instant agent responses.


In [ ]:
# ❌ WRONG: Polling threads.list() in a loop to detect new emails
# This is inefficient, slow, and misses emails that arrive between polls.
# It also consumes API quota for every empty poll.

import time

print("Polling for new emails... (press Ctrl+C to stop)")
seen_thread_ids = set()

while True:
    threads = client.threads.list(inbox_id=inbox.id, limit=20)

    for thread in threads.data:
        if thread.thread_id not in seen_thread_ids:
            seen_thread_ids.add(thread.thread_id)
            print(f"New thread: {thread.subject}")

            # Now need a SECOND API call just to get the message content
            messages = client.threads.messages(thread.thread_id)
            content = messages[-1].content if messages else ""
            # ... run agent, reply, etc.

    time.sleep(10)   # wasted 10 seconds of customer wait time every cycle


In [ ]:
# ✅ CORRECT: Use webhooks — Commune pushes each email to your endpoint instantly
# No polling, no wasted API calls, no latency.

# Step 1: Write a webhook handler (FastAPI example)
webhook_example = '''
import json, os
from fastapi import FastAPI, Request, Response
from commune import CommuneClient, verify_signature, WebhookVerificationError

app = FastAPI()
client = CommuneClient(api_key=os.environ["COMMUNE_API_KEY"])

@app.post("/webhook")
async def handle_email(request: Request):
    body = await request.body()   # raw bytes — required for HMAC

    try:
        verify_signature(
            payload=body,
            signature=request.headers["x-commune-signature"],
            secret=os.environ["COMMUNE_WEBHOOK_SECRET"],
            timestamp=request.headers["x-commune-timestamp"],
        )
    except WebhookVerificationError as e:
        return Response(status_code=401, content=str(e))

    payload = json.loads(body)
    thread_id = payload["thread_id"]   # no second API call needed
    content   = payload["content"]
    sender    = payload["sender"]

    # Run your agent, then reply in the same thread
    reply = your_agent.run(content)
    client.messages.send(
        to=sender, text=reply,
        inbox_id=payload["inboxId"],
        thread_id=thread_id,           # always pass thread_id for replies
    )
    return {"ok": True}
'''

print("Webhook handler (copy into your server file):")
print(webhook_example)

# Step 2: For local development, use ngrok to expose localhost
print()
print("Expose your local server:")
print("  ngrok http 8000")
print()
print("Then register the ngrok URL in the Commune dashboard:")
print("  Dashboard → Inbox → Webhooks → Add endpoint")
print("  URL: https://<your-ngrok-id>.ngrok-free.app/webhook")


## What's next?

You've got the basics. Here's what to explore next:

- **[Semantic search](../capabilities/semantic-search/)** — search thread history with natural language
- **[Structured extraction](../capabilities/structured-extraction/)** — automatically parse email fields to JSON
- **[SMS](../capabilities/sms/)** — provision a phone number and send SMS
- **[LangChain integration](../langchain/)** — wrap Commune as LangChain tools
- **[CrewAI integration](../crewai/)** — multi-agent email coordination

**Commune docs:** https://commune.email/docs  
**GitHub:** https://github.com/shanjai-raj/commune-cookbook